# DBOX exploration
**Things I want to know as an investor**

## Metrics
1) Seat occupancy rate. The biggest question with DBOX is whether it's a gimmick or has real staying power. A continuous occupancy decline among long-adopted theaters, sustained across multiple location is a sign that this is a dead-end tech with no consumer adoption. On the contrary, a higher utilization rate is one of the best alphas. It can also estimate Quarterly revenue - any dislocation is alpha
2) Seat installation cadence
3) Number of screens installed. We know DBOX is a partial upgrade - this is like a growth ceiling
4) Overall box-office health

## Data architecture        
This is an analytical database. Some downstream processing is acceptable.
Design considerations:
* Avoid window functions in case we move to incremental refreshses in the future
* How granular should it be?
* Reusable
  

# Setup

In [9]:
from databricks.connect.session import DatabricksSession as SparkSession
import pyspark.sql.functions as F
from pyspark.sql.window import *
from datetime import datetime, timedelta
import yaml

spark = SparkSession.builder.profile("main").serverless().getOrCreate()

In [ ]:
showtimes=spark.read.table('fandango_dev.base.showtimes')
showtimes.show()

+---------+--------------------+-------------------+------+-----------------+-------+----------------+--------------------+----------+--------+---------------------+----------------+-------------------------+--------------------+------------------+-------+------------+--------------+
|       id|    showtimeHashCode|      ticketingDate|  time|hasMatineeMessage|message|screenReaderTime|ticketingJumpPageURL|theater_id|movie_id|amenityGroupClassName|filmFormatHeader|filmFormatHeaderClassName|         amenity_ids|hasReservedSeating|isDolby|lateNightMsg|movieVariantId|
+---------+--------------------+-------------------+------+-----------------+-------+----------------+--------------------+----------+--------+---------------------+----------------+-------------------------+--------------------+------------------+-------+------------+--------------+
|543210832|v2-b57cc46e005326...|2026-06-02 22:05:00| 3:05p|             true|   NULL|         3:05 PM|https://tickets.f...|     AAFPI|  242515| s

In [22]:
amenities=spark.read.table('fandango_dev.base.amenities_new')
amenities.show()

+----+--------------------+--------------------+--------------------+
|  id|         description|            imageUrl|                name|
+----+--------------------+--------------------+--------------------+
|1204|Music Video celeb...|                NULL|       Dance Parties|
|1214|Cinépolis Junior ...|                NULL|              Junior|
|1225|We don't just wat...|                NULL|       TerrorTuesday|
|1389|High frame rate (...|                NULL|              HFR 3D|
|1521|Recline beyond st...|                NULL|           Max Relax|
|1539|This feature is p...|                NULL|             HDR 300|
|1196|This film is pres...|                NULL|    Kannada Language|
|1264|Family Parties in...|                NULL|        Family Party|
|1321|Sit back, relax, ...|                NULL|  Dine-In Experience|
|1401|This film is pres...|                NULL|Kannada with Engl...|
|1517|This showtime fea...|                NULL|          Discussion|
|1424|Breakfast incl

In [18]:
theaters = spark.read.table('fandango_dev.base.theaters')
theaters.show(5)

+--------------+--------------------+--------------------+----------+--------------------+-------------+
|franchise_name|        theater_name|         theater_url|theater_id|         snapshot_ts|_rescued_data|
+--------------+--------------------+--------------------+----------+--------------------+-------------+
|           AMC|       AMC Summit 16|https://www.fanda...|     aahln|2026-09-17 08:01:...|         NULL|
|           AMC|AMC CLASSIC Markt...|https://www.fanda...|     aatlb|2026-09-17 08:01:...|         NULL|
|           AMC|AMC CLASSIC Jubil...|https://www.fanda...|     aaqnj|2026-09-17 08:01:...|         NULL|
|           AMC|AMC CLASSIC Decat...|https://www.fanda...|     aawyc|2026-09-17 08:01:...|         NULL|
|           AMC|AMC CLASSIC Dothan 6|https://www.fanda...|     aawks|2026-09-17 08:01:...|         NULL|
+--------------+--------------------+--------------------+----------+--------------------+-------------+
only showing top 5 rows


In [0]:
smap = spark.read.table('fandango_dev.scratchpad.latestSeatmaps')
smap.display()

showtimeId snapshot_ts _corrupt_record auditoriumId backgroundHeight backgroundWidth chainCode id infoBoxes mapOffsetX mapOffsetY maxTicketLimit seatBlocks seats surcharges theaterId theaterName tmsId totalAvailableSeatCount totalHeight totalSeatCount totalWidth run_id rn null 1787623690 <!DOCTYPE html> null null null null null null null null null null null null null null null null null null null 1787623690 1 513228879 1787623690 null 2 1600.0 1600.0 MEGA A5mZeLb3 List(List(, sb-4862, , , Both seats must be purchased together, Paired Seating), List(, sb-4863, , , Both seats must be purchased together, Paired Seating), List(, sb-4864, , , Both seats must be purchased together, Paired Seating), List(, sb-4865, , , Both seats must be purchased together, Paired Seating), List(, sb-4866, , , Both seats must be purchased together, Paired Seating), List(, sb-4867, , , Both seats must be purchased together, Paired Seating)) 156.7 337.5 10 List(List(4862, Paired Seating, 2, false, twoseater), List(4863, Paired Seating, 2, false, twoseater), List(4864, Paired Seating, 2, false, twoseater), List(4865, Paired Seating, 2, false, twoseater), List(4866, Paired Seating, 2, false, twoseater), List(4867, Paired Seating, 2, false, twoseater)) List(List(reserved0000000002, 0000000002, List(null, null), null, 8, 42.2, A14, null, , , 1, null, false, A, standard, 42.2, 267.8, 0.0), List(reserved0000000002, 0000000002, List(null, null), null, 9, 42.2, A13, null, , , 1, null, false, A, standard, 42.2, 326.4, 0.0), List(reserved0000000002, 0000000002, List(null, null), null, 10, 42.2, A12, null, , , 1, null, false, A, standard, 42.2, 384.9, 0.0), List(reserved0000000002, 0000000002, List(null, null), null, 11, 42.2, A11, null, , , 1, null, false, A, standard, 42.2, 443.7, 0.0), List(reserved0000000002, 0000000002, List(null, null), null, 12, 42.2, A10, null, , , 1, null, false, A, standard, 42.2, 502.3, 0.0), List(reserved0000000002, 0000000002, List(null, null), null, 13, 42.2, A9, null, , , 1, null, false, A, standard, 42.2, 560.9, 0.0), List(reserved0000000002, 0000000002, List(null, null), null, 14, 42.2, A8, null, , , 1, null, false, A, standard, 42.2, 619.4, 0.0), List(reserved0000000002, 0000000002, List(null, null), null, 15, 42.2, A7, null, , , 1, null, false, A, standard, 42.2, 678.0, 0.0), List(reserved0000000002, 0000000002, List(null, null), null, 16, 42.2, A6, null, , , 1, null, false, A, standard, 42.2, 736.8, 0.0), List(reserved0000000002, 0000000002, List(null, null), null, 17, 42.2, A5, null, , , 1, null, false, A, standard, 42.2, 795.4, 0.0), List(reserved0000000002, 0000000002, List(null, null), null, 18, 42.2, A4, null, , , 1, null, false, A, standard, 42.2, 853.9, 0.0), List(reserved0000000002, 0000000002, List(null, null), null, 19, 42.2, A3, null, , , 1, null, false, A, standard, 42.2, 912.5, 0.0), List(reserved0000000002, 0000000002, List(null, null), null, 20, 42.2, A2, null, , , 1, null, false, A, standard, 42.2, 971.1, 0.0), List(reserved0000000002, 0000000002, List(null, null), null, 21, 42.2, A1, null, , , 1, null, false, A, standard, 42.2, 1029.9, 0.0), List(reserved0000000002, 0000000002, List(null, null), null, 6, 42.2, B18, null, , , 2, null, false, A, standard, 42.2, 155.2, 111.7), List(reserved0000000002, 0000000002, List(null, null), null, 7, 42.2, B17, null, , , 2, null, false, A, standard, 42.2, 213.7, 111.7), List(reserved0000000002, 0000000002, List(null, null), null, 8, 42.2, B16, null, , , 2, null, false, A, standard, 42.2, 272.5, 111.7), List(reserved0000000002, 0000000002, List(null, null), null, 9, 42.2, B15, null, , , 2, null, false, A, standard, 42.2, 331.1, 111.7), List(reserved0000000002, 0000000002, List(null, null), null, 10, 42.2, B14, null, , , 2, null, false, A, standard, 42.2, 389.7, 111.7), List(reserved0000000002, 0000000002, List(null, null), null, 11, 42.2, B13, null, , , 2, null, false, A, standard, 42.2, 448.2, 111.7), List(reserved0000000002, 0000000002, List(null, null), null, 12, 42.2, B1

In [19]:
seatreservationhistory = spark.read.table('fandango_dev.base.seatreservationhistory')

In [37]:
dbox_amenity = (
    amenities
    .filter(
        f.upper(f.col('name')).contains('D-BOX') 
        | f.upper(f.col('name')).contains('DBOX')
    )
)

dbox_filter.show()

+----+--------------------+--------+------------------+
|  id|         description|imageUrl|              name|
+----+--------------------+--------+------------------+
|1027|The D-BOX experie...|    NULL|             D-Box|
|1466|D-BOX (some seats...|    NULL|D-BOX (some seats)|
|1484|D-BOX's cinematic...|    NULL|      D-Box Luxury|
+----+--------------------+--------+------------------+



In [48]:
from pyspark.sql import Window

dbox_showtimes = (
    showtimes
    .select(f.col('id').alias('showtimeId'), 'showtimeHashCode', 'theater_id', f.expr('explode(amenity_ids) as amenity_id'))
    .join(dbox_amenity, on=f.col('amenity_id') == dbox_amenity.id, how='inner')
    .withColumn('rn', f.row_number().over(Window.partitionBy('showtimeHashCode').orderBy(f.lit(1))))
    .filter(f.col('rn') ==  1)
)

dbox_showtimes.show()

+----------+--------------------+----------+----------+----+--------------------+--------+-----+---+
|showtimeId|    showtimeHashCode|theater_id|amenity_id|  id|         description|imageUrl| name| rn|
+----------+--------------------+----------+----------+----+--------------------+--------+-----+---+
| 550166555|v2-0003e479ef3c6c...|     AAVPP|      1027|1027|The D-BOX experie...|    NULL|D-Box|  1|
| 539994784|v2-00056471c8d38b...|     AAVIT|      1027|1027|The D-BOX experie...|    NULL|D-Box|  1|
| 534991577|v2-0005971757173d...|     AAUNK|      1027|1027|The D-BOX experie...|    NULL|D-Box|  1|
| 562204795|v2-0005ba4366295b...|     AAIQK|      1027|1027|The D-BOX experie...|    NULL|D-Box|  1|
| 560766552|v2-000616c362fa41...|     AAHCV|      1027|1027|The D-BOX experie...|    NULL|D-Box|  1|
| 548417742|v2-000653836f9720...|     AAYVY|      1027|1027|The D-BOX experie...|    NULL|D-Box|  1|
| 561041411|v2-00069b5de5c958...|     AACKT|      1027|1027|The D-BOX experie...|    NULL|D

In [54]:
dbox_theaters = (
    dbox_showtimes
    .select('theater_id')
    .distinct()
)

dbox_franchises = (
    dbox_theaters
    .join(theaters, on=f.upper(dbox_theaters.theater_id) == f.upper(theaters.theater_id))
    .groupBy('franchise_name')
    .agg(f.count(f.lit(1)).alias('count'))
    .orderBy(f.col('count').desc())
)

dbox_franchises.show()

+--------------------+-----+
|      franchise_name|count|
+--------------------+-----+
|   Cinemark Theatres|  142|
|     Galaxy Theatres|   13|
|         CMX Cinemas|   10|
|   Megaplex Theatres|    6|
|         Cinema West|    5|
|    Premiere Cinemas|    4|
|    Cinelux Theatres|    2|
|   Santikos Theaters|    1|
|      Zurich Cinemas|    1|
|D'Place Entertain...|    1|
| Celebration Cinema |    1|
|    Chinese Theatres|    1|
|    Phoenix Theatres|    1|
|The Oaks Center C...|    1|
|          GQT Movies|    1|
|        B&B Theaters|    1|
|Picture Show Ente...|    1|
|Metropolitan Thea...|    1|
| Contra Costa Cinema|    1|
|   UltraStar Cinemas|    1|
+--------------------+-----+
only showing top 20 rows


# DBOX


In [57]:
# Q. How many DBOX seats were available on day A?

from pyspark.sql import Window

dbox_showtimes = (
    showtimes
    .select(f.col('id').alias('showtimeId'), 'showtimeHashCode', 'theater_id', f.expr('explode(amenity_ids) as amenity_id'))
    .join(dbox_filter, on=f.col('amenity_id') == dbox_filter.id, how="inner")
    .withColumn('rn', f.row_number().over(Window.partitionBy('showtimeHashCode').orderBy(f.lit(1))))
    .filter(f.col('rn') == 1)
)

dbox_showtimes.show(5)

+----------+--------------------+----------+----------+----+--------------------+--------+-----+---+
|showtimeId|    showtimeHashCode|theater_id|amenity_id|  id|         description|imageUrl| name| rn|
+----------+--------------------+----------+----------+----+--------------------+--------+-----+---+
| 550166555|v2-0003e479ef3c6c...|     AAVPP|      1027|1027|The D-BOX experie...|    NULL|D-Box|  1|
| 539994784|v2-00056471c8d38b...|     AAVIT|      1027|1027|The D-BOX experie...|    NULL|D-Box|  1|
| 534991577|v2-0005971757173d...|     AAUNK|      1027|1027|The D-BOX experie...|    NULL|D-Box|  1|
| 562204795|v2-0005ba4366295b...|     AAIQK|      1027|1027|The D-BOX experie...|    NULL|D-Box|  1|
| 560766552|v2-000616c362fa41...|     AAHCV|      1027|1027|The D-BOX experie...|    NULL|D-Box|  1|
+----------+--------------------+----------+----------+----+--------------------+--------+-----+---+
only showing top 5 rows


# Cinemark adoption

## How many Cinemark [theaters](url) have adoped DBOX?

In [58]:
cinemark_theaters = (
    theaters
    .filter('franchise_name = "Cinemark Theatres"')
)


num_cinemark_theaters  = cinemark_theaters.count()
num_cinemark_theaters_with_dbox = dbox_franchises.filter('franchise_name = "Cinemark Theatres"').collect()[0][1]

print(f'Cinemark has {num_cinemark_theaters} total theaters in the US, of which {num_cinemark_theaters_with_dbox} has DBOX installed, representing a {np.round(100*(num_cinemark_theaters_with_dbox / num_cinemark_theaters), 1)}% install rate')

Cinemark has 304 total theaters in the US, of which 142 has DBOX installed, representing a 46.7% install rate


## How many DBOX screens does Cinemark have?

In [76]:
sRH = spark.read.table('fandango_dev.base.seatReservationHistory')

cinemark_seatmaps = (
    theaters
    .filter('franchise_name = "Cinemark Theatres"').alias('theaters')
    .join(showtimes.alias('showtimes'), on=f.upper(theaters.theater_id)==f.upper(f.col('showtimes.theater_id')))
    .join(
        sRH.withColumnsRenamed({'id': 'screen_id'}).alias('sRH'),
        on=f.col('showtimes.id') == f.col('sRH.showtimeId')
    )
)

dbox_seatmaps = (
    cinemark_seatmaps
    .join(dbox_showtimes, on='showtimeId', how='inner')
)

num_cinemark_screens = cinemark_seatmaps.select('screen_id').distinct().count()
num_cinemark_screens_with_dbox = dbox_seatmaps.select('screen_id').distinct().count()
print(f"Cinemark has a total of {num_cinemark_screens} screens, of which {num_cinemark_screens_with_dbox} support DBOX, or {np.round(100 * num_cinemark_screens_with_dbox / num_cinemark_screens,1)}% adoption rate by screen count")

Cinemark has a total of 4756 screens, of which 495 support DBOX, or 10.4% adoption rate by screen count


In [77]:
num_cinemark_seats = (
    dbox_seatmaps
    .withColumn('rn1', f.row_number().over(Window.partitionBy('screen_id').orderBy(f.lit(1))))
    .filter('rn1=1')
    .agg(f.sum('totalAvailableSeatCount'))
    .collect()
)[0][0]

print(f"Cinemark has a total of {num_cinemark_seats} DBOX seats")

UnknownException: (DENY_NEW_AND_EXISTING_RESOURCES) BAD_REQUEST: Cannot create the resource, please try again later.